# Statistics you can explain · Student

Pandas, Clearly · Pandas for Data Science · Version 1.0.0

Dr. Ali Al Bataineh  
Computer Information Science  
Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates

Read, predict, run, change, and explain. Every worked example starts from fresh teaching data. The original student/café tables are fictional; the public Penguins files retain their documented source.

Run the setup cell first. Keep this notebook inside the downloaded course folder so its `data/` folder can be found. Exports go to `outputs/`. For public-data lessons, extract the full notebook practice pack.

Questions: What does one row represent? Which files are inputs, and where will your outputs be written?

In [ ]:
# Pandas, Clearly
# Dr. Ali Al Bataineh
# Computer Information Science
# Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates
# Run once before the lessons. It prepares explicitly fictional practice data.
# Questions: What does 'as pd' mean? Where will exported files be saved?
# Try: print(pd.__version__) and print(OUTPUT_DIR.resolve()).
# Explain: Why should source data and analysis outputs use different folders?
import os
import tempfile
from pathlib import Path
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "pandas-clearly-mpl"))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import sqlite3
from pathlib import Path
from io import StringIO

# Find data from the repository root or either notebook subdirectory.
# Browser workers mount the same data tree at /course/data.
COURSE_ROOT = next(
    (parent for parent in [Path.cwd(), *Path.cwd().parents]
     if (parent / "data" / "students.csv").is_file()),
    Path.cwd(),
)
DATA_DIR = COURSE_ROOT / "data"
OUTPUT_DIR = COURSE_ROOT / "outputs"
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

_STUDENT_ROWS = [
    ["S001", "Amira", "RAK", "AI", 20, 8, 88, 95, "2026-09-01"],
    ["S002", "Bilal", "Dubai", "IT", 21, 5, 72, 85, "2026-09-02"],
    ["S003", "Chloe", "RAK", "AI", 19, 9, 91, 98, "2026-09-03"],
    ["S004", "Dana", "Dubai", "IT", 22, 4, 65, 78, "2026-09-04"],
    ["S005", "Ethan", "RAK", "IT", 20, 6, 79, 90, "2026-09-05"],
    ["S006", "Fatima", "Dubai", "AI", 21, 10, 95, 99, "2026-09-06"],
    ["S007", "Grace", "RAK", "AI", 23, 7, 84, 92, "2026-09-07"],
    ["S008", "Hassan", "Dubai", "IT", 20, 3, 58, 70, "2026-09-08"],
    ["S009", "Ivy", "RAK", "IT", 22, 6, 76, 88, "2026-09-09"],
    ["S010", "Jamal", "Dubai", "AI", 19, 8, 89, 96, "2026-09-10"],
    ["S011", "Khalid", "RAK", "AI", 24, 5, 68, 82, "2026-09-11"],
    ["S012", "Lina", "Dubai", "IT", 21, 7, 82, 94, "2026-09-12"],
]
_STUDENT_COLUMNS = ["Student_ID", "Name", "Campus", "Program", "Age",
                    "Study_Hours", "Score", "Attendance", "Join_Date"]

def students():
    """Return a new, clean practice DataFrame, never real student records."""
    return pd.DataFrame(_STUDENT_ROWS, columns=_STUDENT_COLUMNS)

def records():
    """Return the same practice records as a list of dictionaries."""
    return students().to_dict(orient="records")

def raw_students():
    """Return a fresh table with deliberate data-quality problems."""
    df = students()
    df["Score"] = df["Score"].astype(object)
    df["Study_Hours"] = df["Study_Hours"].astype(object)
    df.loc[0, "Name"] = " Amira "
    df.loc[1, "Campus"] = "dubai"
    df.loc[3, "Score"] = None
    df.loc[5, "Study_Hours"] = None
    df.loc[7, "Score"] = "absent"
    df.loc[10, "Attendance"] = 105
    return pd.concat([df, df.iloc[[2]]], ignore_index=True)

def departments():
    return pd.DataFrame({
        "Program": ["AI", "IT"],
        "Department": ["Computing", "Information Systems"],
        "Advisor": ["Dr. Noor", "Dr. Sami"],
    })

def assessment():
    return pd.DataFrame({
        "Student_ID": ["S001", "S002", "S003", "S004", "S005", "S006",
                       "S007", "S008", "S009", "S010", "S011", "S013"],
        "Quiz": [17, 14, 19, 13, 16, 19, 17, 11, 15, 18, 14, 16],
        "Project": [44, 36, 46, 34, 40, 47, 42, 30, 38, 45, 35, 41],
    })

def attendance():
    return pd.DataFrame({
        "Date": ["2026-10-01", "2026-10-02", "2026-10-03",
                 "2026-10-05", "2026-10-06", "2026-10-07"],
        "Visitors": [20, 24, 18, 30, 28, 32],
        "Campus": ["RAK"] * 6,
    })

# Create missing practice files. Existing files are preserved.
for _name, _frame in [
    ("students.csv", students()), ("raw_students.csv", raw_students()),
    ("departments.csv", departments()), ("assessment.csv", assessment()),
    ("attendance.csv", attendance()),
]:
    if not (DATA_DIR / _name).exists():
        _frame.to_csv(DATA_DIR / _name, index=False)
if not (DATA_DIR / "students.json").exists():
    students().to_json(DATA_DIR / "students.json", orient="records", indent=2)
if not (DATA_DIR / "students.xlsx").exists():
    try:
        students().to_excel(DATA_DIR / "students.xlsx", sheet_name="Students", index=False)
    except ImportError:
        print("Excel needs openpyxl. Install it before Module 20's Excel cell.")

pd.set_option("display.max_rows", 15)
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 110)
plt.rcParams.update({"figure.figsize": (7, 4), "axes.spines.top": False,
                     "axes.spines.right": False, "figure.dpi": 110})
print("Ready. pandas", pd.__version__)
print("Practice data: 12 fictional students. Every helper returns a fresh table.")
print("Exports go to:", OUTPUT_DIR.resolve())


## Module 15. Statistics you can explain

Summarize center, spread, completeness, and association with clear interpretations.

Before you start: You can select numeric columns and identify missing values.

Official reference: https://pandas.pydata.org/docs/user_guide/basics.html#descriptive-statistics

### M15C01. Mean and median respond differently

The mean uses every numeric value in an arithmetic average. The median is the middle value after sorting. Comparing them can help you notice how an unusually large or small observation affects a summary.

**Read:** What calculation produces the mean here?

Your explanation: _write here_.

**Predict:** What is the median?

Your explanation: _write here_.

**Explain:** Does a mean equal to the median prove the data are correct?

Your explanation: _write here_.



In [ ]:
# Predict before running: What is the median?
costs = pd.Series([10, 20, 30, 40, 50])  # Fictional costs in AED
result = pd.Series({"Mean_AED": costs.mean(), "Median_AED": costs.median()})
print(result)

**Your turn:** Change the last cost from 50 to 500. Recalculate mean and median in result.

For the check, keep these result names: `result`.

**Hint:** Recreate the Series with the changed final value.

**Watch for this:** An extreme value is not automatically a mistake; investigate its source and meaning.

In [ ]:
# Change this code to complete the task.
# Question: Change the last cost from 50 to 500. Recalculate mean and median in result.
costs = pd.Series([10, 20, 30, 40, 50])  # Fictional costs in AED
result = pd.Series({"Mean_AED": costs.mean(), "Median_AED": costs.median()})
print(result)

**Quick check:** Which summary stays 30 after this change?

1. Mean
2. Median
3. Maximum

**Explain what you learned:** Explain which summary fits the question and distribution.

### M15C02. Spread and the standard-deviation convention

Standard deviation measures spread around the mean in the same units as the values. pandas uses sample standard deviation by default, with ddof=1. ddof=0 uses the population-style denominator; choose based on what the data represent.

**Read:** What does ddof change?

Your explanation: _write here_.

**Predict:** Which standard deviation is larger for these data?

Your explanation: _write here_.

**Explain:** Would changing hours to minutes leave the standard deviation numerically unchanged?

Your explanation: _write here_.



In [ ]:
# Predict before running: Which standard deviation is larger for these data?
hours = pd.Series([2.0, 4.0, 6.0])
result = pd.Series({"Mean_hours": hours.mean(), "Sample_std_hours": hours.std(ddof=1), "Population_std_hours": hours.std(ddof=0)})
print(result.round(4))

**Your turn:** Replace the hours with [4.0, 4.0, 4.0]. Recalculate both standard deviations.

For the check, keep these result names: `result`.

**Hint:** A set of identical values has no spread around its mean.

**Watch for this:** With only one known value, sample std is missing because n - 1 is zero.

In [ ]:
# Change this code to complete the task.
# Question: Replace the hours with [4.0, 4.0, 4.0]. Recalculate both standard deviations.
hours = pd.Series([2.0, 4.0, 6.0])
result = pd.Series({"Mean_hours": hours.mean(), "Sample_std_hours": hours.std(ddof=1), "Population_std_hours": hours.std(ddof=0)})
print(result.round(4))

**Quick check:** Which ddof does pandas std() use by default?

1. 0
2. 1
3. The number of rows

**Explain what you learned:** State the spread measure and its convention.

### M15C03. Describe, count, and quantiles

`describe()` summarizes a numeric Series with count, mean, spread, and selected percentiles. Count excludes missing values; size counts all entries. A quantile describes a position in the distribution, and its value may be interpolated between observations.

**Read:** Does describe count the missing score as an observed score?

Your explanation: _write here_.

**Predict:** What is the 50% value?

Your explanation: _write here_.

**Explain:** Why does a reported average need a known-value count?

Your explanation: _write here_.



In [ ]:
# Predict before running: What is the 50% value?
scores = pd.Series([60.0, 70.0, np.nan, 80.0, 90.0])
result = scores.describe()
print(result)
print("Rows:", scores.size, "Known:", scores.count())

**Your turn:** For the same scores, report the 25th, 50th, and 75th percentiles explicitly using quantile.

For the check, keep these result names: `result`.

**Hint:** Pass a list of probabilities [0.25, 0.5, 0.75].

**Watch for this:** Describe is a starting point. It does not check whether IDs, units, ranges, or collection procedures are valid.

In [ ]:
# Change this code to complete the task.
# Question: For the same scores, report the 25th, 50th, and 75th percentiles explicitly using quantile.
scores = pd.Series([60.0, 70.0, np.nan, 80.0, 90.0])
result = scores.describe()
print(result)
print("Rows:", scores.size, "Known:", scores.count())

**Quick check:** Which quantity includes a missing entry?

1. Series.count()
2. Series.size
3. The count row of describe()

**Explain what you learned:** Report completeness alongside summary statistics.

### M15C04. Correlation describes association

`corr()` computes pairwise correlations between numeric variables; its default method is Pearson correlation. Positive values indicate that higher values tend to occur together in a linear pattern. Correlation does not establish that one variable causes the other.

**Read:** Why are only three columns selected before corr()?

Your explanation: _write here_.

**Predict:** What values appear on the diagonal?

Your explanation: _write here_.

**Explain:** Does a positive Study_Hours-Score correlation prove that studying longer caused these scores?

Your explanation: _write here_.



In [ ]:
# Predict before running: What values appear on the diagonal?
df = students()
result = df[["Study_Hours", "Score", "Attendance"]].corr(method="pearson")
print(result.round(3))

**Your turn:** Calculate Spearman rank correlation for Study_Hours and Score. Store the two-by-two matrix in result.

For the check, keep these result names: `result`.

**Hint:** Change method to "spearman" after selecting the two columns.

**Watch for this:** Missing entries are handled pairwise, so different correlations can use different numbers of observations.

In [ ]:
# Change this code to complete the task.
# Question: Calculate Spearman rank correlation for Study_Hours and Score. Store the two-by-two matrix in result.
df = students()
result = df[["Study_Hours", "Score", "Attendance"]].corr(method="pearson")
print(result.round(3))

**Quick check:** What claim does correlation alone justify?

1. A causal effect has been proved
2. An association was measured in the available data
3. The model will predict perfectly

**Explain what you learned:** Interpret a correlation as an association with limitations.

## Keep practicing

Can you apply this idea to a different table without copying the example?

Dr. Ali Al Bataineh  
Computer Information Science  
Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates